In [2]:
import pandas as pd 
import numpy as np
import pynapple as nap

In [1]:

from dandi.download import download as dandi_download
import os

filenames = ["sub-Dataset-3-Animals-1-to-4",
             "sub-Dataset-4-Animal-2-sess-1-to-7",
             "sub-Dataset-4-Animal-2-sess-8-to-12",
             "sub-Dataset-4-Animal-3-sess-1-to-6",
             "sub-Dataset-4-Animal-3-sess-7-to-11",
             "sub-Dataset-5-Animal-1",
             "sub-Dataset-5-Animals-2-ses-1-to-19",
             "sub-Dataset-5-Animals-2-ses-20-to-39",
             "sub-Dataset-5-Animals-2-ses-40-to-51",
             "sub-Dataset-5-Animals-3&4"]
# Set the URL for the DANDI file
url_base = 'https://dandiarchive.org/dandiset/001057/draft/files?location='

for filename in filenames:
  url = url_base + filename
  # Download the file into the current working directory
  # It will skip downloading any files you've already downloaded
  dandi_download([url], output_dir = os.path.join(os.getcwd(),"data"),
                 existing = "skip")

PATH                                                          SIZE    DONE    DONE% CHECKSUM STATUS          MESSAGE
sub-Dataset-3-Animals-1-to-4\sub-Dataset-3-Animals-1-to-4.nwb 80.5 MB 80.5 MB  100%    ok    done                   
Summary:                                                      80.5 MB 80.5 MB                1 done                 
                                                                      100.00%                                       
PATH                                                                      SIZE     DONE           DONE% CHECKSUM STATUS          MESSAGE
sub-Dataset-4-Animal-2-sess-1-to-7\sub-Dataset-4-Animal-2-sess-1-to-7.nwb 379.5 MB 379.5 MB        100%    ok    done                   
Summary:                                                                  379.5 MB 379.5 MB                      1 done                 
                                                                                   100.00%                               

In [3]:
# function from neuro task github https://github.com/catniplab/NeuroTask/blob/main/tutorial_data_analysis_nwb.ipynb
def get_dataframe(data, filter_result=False):
    """
    Load a Nwb file, apply filters if provided, and return the filtered DataFrame and bin size.

    Parameters:
    data (str): Path to the nwb file.
    filter_letters (list, optional): List of letters to filter out from the 'result' column.

    Returns:
    tuple: Filtered DataFrame and bin size (in milliseconds) as a float.
    """
    bin = 1000/data.nwb.processing['spikes'].data_interfaces['spikes_counts'].rate

    keys = list(data.keys())
    dataframes = []

    for key in keys:
        if key == 'spikes_counts':
            # Create DataFrame for 'spikes_counts' with 'Neuron' prefix
            sp = pd.DataFrame(data['spikes_counts'].values, columns=data['spikes_counts'].columns)
            sp.columns = ['Neuron' + str(col) for col in sp.columns]
            dataframes.append(sp)
        else:
            df = pd.DataFrame(data[key].values, columns=[key])
            dataframes.append(df)


    # Concatenate all DataFrames into a single DataFrame
    final_df = pd.concat(dataframes, axis=1)
    print(f'Data loaded with bin size of {bin:.1f} ms')

    # Convert 'result' column from byte string to regular string if it exists
    if 'result' in final_df.columns and final_df['result'].dtype == object:
        # Check if the values are indeed byte strings before decoding
        if len(final_df['result']) > 0 and isinstance(final_df['result'].iloc[0], bytes):
            final_df['result'] = final_df['result'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

    # Also convert 'brain_region' if it exists and contains byte strings
    if 'brain_region' in final_df.columns and final_df['brain_region'].dtype == object:
        if len(final_df['brain_region']) > 0 and isinstance(final_df['brain_region'].iloc[0], bytes):
            final_df['brain_region'] = final_df['brain_region'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)


    if filter_result:
        return final_df[final_df['result'].isin(filter_result)], bin
    else:
        return final_df, bin

In [5]:
# check bin size consistency
path_base = os.path.join(os.getcwd(),"data")

bs = []
for filename in filenames:
  fpath = os.path.join(path_base,filename,filename+'.nwb')
  dat = nap.load_file(fpath)
  df, bin = get_dataframe(dat)
  bs.append(bin)
  # check if we have brain region info
  if 'brain_region' in df.columns:
    print(f"{filename} has brain region info")


Data loaded with bin size of 30.0 ms
sub-Dataset-3-Animals-1-to-4 has brain region info
Data loaded with bin size of 1.0 ms
Data loaded with bin size of 1.0 ms
Data loaded with bin size of 1.0 ms
Data loaded with bin size of 1.0 ms
Data loaded with bin size of 10.0 ms
Data loaded with bin size of 10.0 ms
Data loaded with bin size of 10.0 ms
Data loaded with bin size of 10.0 ms
Data loaded with bin size of 10.0 ms
[np.float64(29.999999999999996), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(10.0), np.float64(10.0), np.float64(10.0), np.float64(10.0), np.float64(10.0)]


In [25]:
filename = r"C:\Users\lilytong\Documents\GitHub\NMA-shichimi-rulebreakerhalf-comparing-network\data\sub-Dataset-3-Animals-1-to-4\sub-Dataset-3-Animals-1-to-4.nwb"
dat = nap.load_file(filename)
dat["brain_region"].data()

<StrDataset for HDF5 dataset "data": shape (1074509,), type "|O">

In [28]:
dat

sub-Dataset-5-Animals-3&4
┍━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━┑
│ Keys              │ Type     │
┝━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━┥
│ spikes_counts     │ TsdFrame │
│ trial_id          │ Tsd      │
│ session           │ Tsd      │
│ datasetID         │ Tsd      │
│ animal            │ Tsd      │
│ result            │ Tsd      │
│ EventTarget_onset │ Tsd      │
│ EventGo_cue       │ Tsd      │
│ targets_dir       │ Tsd      │
│ cursor_vel_y      │ Tsd      │
│ cursor_vel_x      │ Tsd      │
│ cursor_pos_y      │ Tsd      │
│ cursor_pos_x      │ Tsd      │
│ cursor_acc_y      │ Tsd      │
│ cursor_acc_x      │ Tsd      │
│ Target_ID         │ Tsd      │
┕━━━━━━━━━━━━━━━━━━━┷━━━━━━━━━━┙

In [ ]:
filename = r"C:\Users\lilytong\Documents\GitHub\NMA-shichimi-rulebreakerhalf-comparing-network\data\sub-Dataset-5-Animals-2-ses-20-to-39\sub-Dataset-5-Animals-2-ses-20-to-39.nwb"
dat = nap.load_file(filename)
dat.nwb

Data type,object
Shape,"(4,)"
Array size,32.00 bytes
Chunk shape,None
Compression,None
Compression opts,None
Uncompressed size (bytes),32
Compressed size (bytes),64
Compression ratio,0.5
Data type,float64
Shape,"(1777949,)"


In [16]:
filename = r"C:\Users\lilytong\Documents\GitHub\NMA-shichimi-rulebreakerhalf-comparing-network\data\sub-Dataset-5-Animals-2-ses-40-to-51\sub-Dataset-5-Animals-2-ses-40-to-51.nwb"
dat = nap.load_file(filename)
dat.nwb

Data type,object
Shape,"(4,)"
Array size,32.00 bytes
Chunk shape,None
Compression,None
Compression opts,None
Uncompressed size (bytes),32
Compressed size (bytes),64
Compression ratio,0.5
Data type,float64
Shape,"(1054962,)"


In [ ]:
filename = r"C:\Users\lilytong\Documents\GitHub\NMA-shichimi-rulebreakerhalf-comparing-network\data\sub-Dataset-5-Animal-1\sub-Dataset-5-Animal-1.nwb"
dat = nap.load_file(filename)
dat.nwb.subject

subject pynwb.file.Subject at 0x1739465833712
Fields:
  age: P10M
  age__reference: birth
  description: M (animal 1)
  sex: M
  species: Rhesus macaques
  subject_id: Dataset_5_Animal-1

In [27]:
filename = r"C:\Users\lilytong\Documents\GitHub\NMA-shichimi-rulebreakerhalf-comparing-network\data\sub-Dataset-5-Animals-3&4\sub-Dataset-5-Animals-3&4.nwb"
dat = nap.load_file(filename)
dat.nwb.subject

subject pynwb.file.Subject at 0x1741331459248
Fields:
  age: P10M
  age__reference: birth
  description: T (animal 3) and J (animal 4)
  sex: M
  species: Rhesus macaques
  subject_id: Dataset_5_Animals-3&4